In [1]:
from data import load, ProteinDataset, ProteinPairDataset, pair_collate_fn


data = load(f'/home/burger/bioinfo/project/pdb.hdf5')
datalib = ProteinDataset(data)
pdb2idx = [(data[5][i], i) for i in range(len(data[5]))] # pdb name -> idx
pdb2idx = dict(pdb2idx)

In [ ]:
from torch.utils.data import DataLoader


pair_dataset = ProteinPairDataset(datalib, '/home/burger/bioinfo/project/tmalign.out', pdb2idx)
loader = DataLoader(pair_dataset, batch_size=256, shuffle=False, collate_fn=pair_collate_fn, num_workers=6)

In [3]:
import torch as pt
import torch.nn as nn
from torch_geometric.data import Data
import torch.nn.functional as F
import torch_geometric.nn as gnn


class EmbeddingBlock(nn.Module):
    def __init__(self, out_channels:int=256):
        super().__init__()
        self.emb = nn.Embedding(num_embeddings=20, embedding_dim=out_channels)

    def forward(self, data):
        graph, seq, _ = data
        node_attr, edge_index, edge_len = graph.x, graph.edge_index, graph.edge_attr
        N = node_attr.size(0)
        # edge_index:[2, N-1+num_nho] edge_attr:[N-1+num_nho]
        tai_idx = pt.stack((edge_index[0, :N-1], edge_index[1, :N-1]), dim=0)
        nho_idx = pt.stack((edge_index[0, N-1:], edge_index[1, N-1:]), dim=0)
        edge_idx = pt.cat((tai_idx, tai_idx.flip(0), nho_idx, nho_idx.flip(0)), dim=0)
        tai_len = edge_len[:N-1]
        nho_len = edge_len[N-1:]
        edge_len = pt.cat((tai_len, tai_len.flip(0), nho_len, nho_len.flip(0)), dim=0)
        # edge_attr : 键长, is_peptide, direction, is_hbond, 肽键：±1 氢键：0
        is_peptide = pt.cat((pt.ones_like(tai_len), pt.ones_like(tai_len), pt.zeros_like(nho_len), pt.zeros_like(nho_len)), dim=0)
        is_hbond = 1 - is_peptide
        direction = pt.cat((pt.ones_like(tai_len), -1*pt.ones_like(tai_len), pt.zeros_like(nho_len), pt.zeros_like(nho_len)), dim=0)
        edge_attr = pt.stack((edge_len, is_peptide, is_hbond, direction), dim=0)
        node_emb = self.emb(seq)
        node_emb = pt.cat((node_emb, node_attr.unsqueeze(-1)), dim=1)
        return Data(x=node_emb, edge_index=edge_idx, edge_attr=edge_attr)

In [4]:
class ProtainGCN(nn.Module):
    def __init__(self, embed_dim:int=256, hidden_channels:int=256, num_layers:int=3, out_channels:int=512):
        super().__init__()
        self.emb = EmbeddingBlock(out_channels=embed_dim)
        self.gcn = gnn.GCN(embed_dim, hidden_channels, num_layers, out_channels,)
        self.linear1 = nn.Linear(out_channels, 1)
        self.linear2 = nn.Linear(out_channels, 1)

    def embbed(self, seq):
        self.eval()
        with pt.no_grad():
            emb = self.emb(seq)
        return emb

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.gcn(x, edge_index, edge_attr, batch)
        x = gnn.global_mean_pool(x, batch)
        tmscore = self.linear1(x)
        tmscore = pt.sigmoid(tmscore)
        seqid = self.linear2(x)
        seqid = pt.sigmoid(seqid)
        return tmscore, seqid

In [5]:
from sklearn.model_selection import train_test_split

gpu = 6
data_map = pt.arange(len(pair_dataset), dtype=pt.int32)
train_map, testmap = train_test_split(data_map, test_size=10240, random_state=42)
train_set = ProteinPairDataset(pair_dataset, train_map)
test_set = ProteinPairDataset(pair_dataset, testmap)
batch_size = 256
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=pair_collate_fn, drop_last=True, num_workers=6)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, collate_fn=pair_collate_fn, num_workers=6)